# Kiafura — ARC-AGI-2 demonstration entry (ARC Prize 2026 Paper Track linkage)

Five frozen operational memories + fail-closed emission gates + the two-output contract,
emitting predictions-or-named-abstention. **Target-blind:** only `train` pairs and `test[].input`
are read. **DEMONSTRATION OUTPUT — the competition score is not an objective.**

Code and memories: CC0 1.0 Universal (public domain). Pipeline is the frozen inference/gate chain with
light driver glue only (`kaggle-demo/provenance.json`: `no_algorithm_modification: true`).

Submission format: `submission.json`, `attempt_1`/`attempt_2` per test input, in input order;
a slot with no real prediction carries the venue's `[[0, 0], [0, 0]]` example fill.

In [ ]:
# Cell 1 — mounts, package discovery (dataset zip is auto-extracted by Kaggle; naming varies)
import zipfile
from pathlib import Path

INPUTS = Path('/kaggle/input')
print('mounts:', sorted(p.name for p in INPUTS.iterdir()))

comp_dirs = [p for p in INPUTS.iterdir()
             if 'arc-prize-2026-arc-agi-2' in p.name or 'arc_prize_2026_arc_agi_2' in p.name]
assert comp_dirs, f'competition data not mounted: {sorted(p.name for p in INPUTS.iterdir())}'
COMP = comp_dirs[0]
challenges = sorted(COMP.glob('*test*challenges*.json'))
print('competition dir:', COMP.name, '| challenges files:', [c.name for c in challenges])

WORK = Path('/kaggle/working')
PKG_ROOT = WORK / 'pkg'
# layout A: kaggle-demo/driver.py somewhere under the mounts (zip auto-extracted)
driver_files = [p for p in INPUTS.rglob('driver.py') if p.parent.name == 'kaggle-demo']
if driver_files:
    PKG_ROOT = driver_files[0].parent.parent
    print('package root (from mount):', PKG_ROOT)
else:
    # layout B: a zip we must extract ourselves
    zips = sorted(p for mount in INPUTS.iterdir() for p in mount.glob('*.zip'))
    assert zips, 'no package found in /kaggle/input'
    print('extracting:', zips[0])
    PKG_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(PKG_ROOT)
    if not (PKG_ROOT / 'kaggle-demo' / 'driver.py').exists():  # zip may nest one dir
        subs = [d for d in PKG_ROOT.iterdir() if (d / 'kaggle-demo' / 'driver.py').exists()]
        assert subs, 'package layout unexpected after extraction'
        PKG_ROOT = subs[0]
assert (PKG_ROOT / 'kaggle-demo' / 'driver.py').exists(), 'kaggle-demo not found'
assert (PKG_ROOT / 'kaggle-submit' / 'make_submission.py').exists(), 'packer not found'
print('package root:', PKG_ROOT)

In [ ]:
# Cell 2 — run the packer as a subprocess (signal.SIGALRM in the frozen driver is
# fragile inside an IPython kernel) and publish /kaggle/working/submission.json
import json, shutil, subprocess, sys

OUT = WORK / 'outputs'
packer = PKG_ROOT / 'kaggle-submit' / 'make_submission.py'
cmd = [sys.executable, str(packer),
       '--package-dir', str(PKG_ROOT / 'kaggle-demo'),
       '--data-dir', str(COMP),
       '--out', str(OUT),
       '--time-budget-s', '36000']   # wall-clock guard: 10h inside the 12h CPU limit
print('running:', ' '.join(cmd))
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout[-4000:])
if res.stderr:
    print('STDERR tail:', res.stderr[-2000:])
assert res.returncode == 0, 'packer failed'
shutil.copy(OUT / 'submission.json', WORK / 'submission.json')
print('submission.json bytes:', (WORK / 'submission.json').stat().st_size)

In [ ]:
# Cell 3 — validation against the venue contract
sub = json.loads((WORK / 'submission.json').read_text())
chal = json.loads(challenges[0].read_text())
assert set(sub) == set(chal), 'task-id mismatch between submission and challenges'
for tid, slots in sub.items():
    assert len(slots) == len(chal[tid]['test']), f'{tid}: slot-count mismatch'
    for s in slots:
        assert set(s) == {'attempt_1', 'attempt_2'}, f'{tid}: missing attempt keys'
print('VALID:', len(sub), 'tasks;', sum(len(v) for v in sub.values()), 'test-input slots')
print(json.dumps(json.loads((OUT / 'manifest.json').read_text()), indent=1))